# GLSKF 实验

In [ ]:
import numpy as np
from scipy.sparse import eye, csr_matrix
from scipy.linalg import inv, khatri_rao
from skimage.metrics import structural_similarity as ssim
from PIL import Image
import matplotlib.pyplot as plt
import os
import pandas as pd
import time

def cov_matern(d, loghyper, x):
    ell = np.exp(loghyper[0])
    sf2 = np.exp(2 * loghyper[1])
    def f(t):
        if d == 1: return 1
        if d == 3: return 1 + t
        if d == 5: return 1 + t * (1 + t / 3)
        if d == 7: return 1 + t * (1 + t * (6 + t) / 15)
    def m(t):
        return f(t) * np.exp(-t)
    dist_sq = ((x[:, None] - x[None, :]) / ell) ** 2
    return sf2 * m(np.sqrt(d * dist_sq))

def bohman(loghyper, x):
    range_ = np.exp(loghyper[0])
    dis = np.abs(x[:, None] - x[None, :])
    r = np.minimum(dis / range_, 1)
    k = (1 - r) * np.cos(np.pi * r) + np.sin(np.pi * r) / np.pi
    k[k < 1e-16] = 0
    k[np.isnan(k)] = 0
    return k

def unfold(tensor, mode):
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order='F')

def fold(mat, dim, mode):
    index = list()
    index.append(mode)
    for i in range(dim.shape[0]):
        if i != mode:
            index.append(i)
    return np.moveaxis(np.reshape(mat, list(dim[index]), order='F'), 0, mode)

def kroneckerMVM(K3, K2, K1, vec, d1, d2, d3):
    temp1 = (K1 @ vec.reshape(d1, d2, d3, order='F').reshape(d1, -1)).reshape(d1, d2, d3)
    temp2 = (K2 @ temp1.transpose(1, 0, 2).reshape(d2, -1)).reshape(d2, d1, d3).transpose(1, 0, 2)
    temp3 = (K3 @ temp2.transpose(2, 0, 1).reshape(d3, -1)).reshape(d3, d1, d2).transpose(1, 2, 0)
    return temp3.ravel(order='F')

def Ap_operatorT(vec, maskT, KrU, KrU_T, Qu, rho, R, M):
    X = vec.reshape(R, M, order='F')
    temp = KrU @ X
    temp *= maskT
    Ap1 = KrU_T @ temp
    Ap2 = rho * (X @ Qu)
    return (Ap1 + Ap2).ravel(order='F')

def cg_factorT(Qu, rho, KrU, mask_matrixT, YR_tilde, priorvalue, max_iter):
    R, M = YR_tilde.shape
    Y_flat = YR_tilde.ravel(order='F')
    x = priorvalue.copy()
    KrU_T = KrU.T
    Ax = Ap_operatorT(x, mask_matrixT, KrU, KrU_T, Qu, rho, R, M)
    r = Y_flat - Ax
    p = r.copy()
    rsold = np.dot(r, r)
    approxE = np.zeros(max_iter)
    for i in range(max_iter):
        Ap = Ap_operatorT(p, mask_matrixT, KrU, KrU_T, Qu, rho, R, M)
        alpha = rsold / np.dot(p, Ap)
        x += alpha * p
        r -= alpha * Ap
        rsnew = np.dot(r, r)
        approxE[i] = np.sqrt(rsnew)
        if approxE[i] < 1e-4:
            break
        p = r + (rsnew / rsold) * p
        rsold = rsnew
    return x, approxE

def Ap_operatorL(vec, pos_obs, Kd, Kt, Ks, gamma, d1, d2, d3):
    x = np.zeros(d1 * d2 * d3)
    x[pos_obs] = vec
    Ap1 = kroneckerMVM(Kd, Kt, Ks, x, d1, d2, d3)
    return Ap1[pos_obs] + gamma * vec

def cg_local(gamma, Kd, Kt, Ks, pos_obs, YR_tilde, priorvalue, max_iter):
    d1, d2, d3 = YR_tilde.shape
    Y_obs = (YR_tilde.ravel(order='F'))[pos_obs]
    x = priorvalue.copy()
    Ax = Ap_operatorL(x, pos_obs, Kd, Kt, Ks, gamma, d1, d2, d3)
    r = Y_obs - Ax
    p = r.copy()
    rsold = np.dot(r, r)
    approxE = np.zeros(max_iter)
    for i in range(max_iter):
        Ap = Ap_operatorL(p, pos_obs, Kd, Kt, Ks, gamma, d1, d2, d3)
        alpha = rsold / np.dot(p, Ap)
        x += alpha * p
        r -= alpha * Ap
        rsnew = np.dot(r, r)
        approxE[i] = np.sqrt(rsnew)
        if approxE[i] < 1e-4:
            break
        p = r + (rsnew / rsold) * p
        rsold = rsnew
    return x, approxE

In [ ]:
def GLSKF(I, Omega, lengthscaleU, lengthscaleR, varianceU, varianceR, tapering_range, d_MaternU, d_MaternR, R, rho, gamma, maxiter, K0, epsilon):
    N = np.array(I.shape)
    D = I.ndim
    maxP = float(np.max(I))

    Omega = Omega.astype(bool)
    pos_miss = np.where(Omega == 0)
    num_obser = np.sum(Omega)
    mask_matrix = [unfold(Omega, d) for d in range(D)]
    idx = np.sum(mask_matrix[2], axis=0) > 0
    train_matrix = I * Omega
    train_matrix = train_matrix[train_matrix > 0]
    Isubmean = I - np.mean(train_matrix)
    T = Isubmean * Omega
    mask_matrixT = [mask_matrix[d].T for d in range(D)]
    mask_flat = [mask_matrix[d].ravel(order='F') for d in range(D)]
    pos_obs = [np.where(mask_flat[d] == 1) for d in range(D)]

    hyper_Ku = [None] * D
    hyper_Ku[0] = [np.log(lengthscaleU[0]), np.log(varianceU[0])]
    hyper_Ku[1] = [np.log(lengthscaleU[1]), np.log(varianceU[1])]
    hyper_Kr = [None] * D
    hyper_Kr[0] = [np.log(lengthscaleR[0]), np.log(varianceR[0]), np.log(tapering_range)]
    hyper_Kr[1] = [np.log(lengthscaleR[1]), np.log(varianceR[1]), np.log(tapering_range)]

    Ku, Kr = [None] * D, [None] * D
    invKu = [None] * D

    x = np.arange(1, N[0] + 1)
    Ku[0] = cov_matern(d_MaternU, hyper_Ku[0], x)
    invKu[0] = inv(Ku[0])
    TaperM = bohman([hyper_Kr[0][2]], x)
    Kr[0] = csr_matrix(cov_matern(d_MaternR, hyper_Kr[0][:2], x) * TaperM)

    x = np.arange(1, N[1] + 1)
    Ku[1] = cov_matern(d_MaternU, hyper_Ku[1], x)
    invKu[1] = inv(Ku[1])
    TaperM = bohman([hyper_Kr[1][2]], x)
    Kr[1] = csr_matrix(cov_matern(d_MaternR, hyper_Kr[1][:2], x) * TaperM)

    invKu[2] = csr_matrix(eye(N[2]))
    Kr[2] = csr_matrix(eye(N[2]))

    X = T
    X[pos_miss] = T.sum() / num_obser
    U = [0.1 * np.random.randn(N[d], R) for d in range(D)]
    M_unfold1 = U[0] @ khatri_rao(U[2], U[1]).T
    M = fold(M_unfold1, N, 0)
    Uvector = [U[d].ravel(order='F') for d in range(D)]
    UTvector = [U[d].T.ravel(order='F') for d in range(D)]
    Rtensor = np.zeros(N)
    Rvector = Rtensor.ravel(order='F')
    Rvector_temp = Rtensor.ravel(order='F')
    X[pos_miss] = M[pos_miss] + Rtensor[pos_miss]

    d_all = np.arange(0, D)
    train_norm = np.linalg.norm(T)
    last_ten = T.copy()
    psnrf = np.zeros(maxiter)
    approxU = [None] * D
    iter = 0
    while True:
        Gtensor = X - Rtensor
        Gtensor_mask = Gtensor * Omega
        for d in range(D):
            dsub = np.delete(d_all, d)
            KrU = khatri_rao(U[dsub[1]], U[dsub[0]])
            HG = KrU.T @ unfold(Gtensor_mask, d).T
            UTvector[d], approxU[d] = cg_factorT(invKu[d], rho, KrU, mask_matrixT[d], HG, UTvector[d], 100)
            U[d] = (UTvector[d].reshape(R, N[d], order='F')).T
        M_unfold1 = U[0] @ (khatri_rao(U[2], U[1]).T)
        M = fold(M_unfold1, N, 0)
        X[pos_miss] = M[pos_miss] + Rtensor[pos_miss]
        if iter >= K0:
            Ltensor = X - M
            Ltensor_mask = Ltensor * Omega
            Rvector_temp[pos_obs[0]], approxE = cg_local(gamma, Kr[2], Kr[1], Kr[0], pos_obs[0],
                                                         Ltensor_mask, Rvector_temp[pos_obs[0]], 100)
            Rvector = kroneckerMVM(Kr[2], Kr[1], Kr[0], Rvector_temp, N[0], N[1], N[2])
            Rtensor = Rvector.reshape(N, order='F')
            Rtensor_unfold3 = unfold(Rtensor, 2)
            Rtensor_unfold3_obs = Rtensor_unfold3[:, idx]
            Kr[2] = np.cov(Rtensor_unfold3_obs)
        else:
            Rtensor = np.zeros_like(Rtensor)
        X[pos_miss] = M[pos_miss] + Rtensor[pos_miss]
        Xori = X + np.mean(train_matrix)
        Xrecovery = np.maximum(0, Xori)
        Xrecovery = np.minimum(maxP, Xrecovery)

        mseC1 = np.linalg.norm(I[:, :, 0].astype(float) - Xrecovery[:, :, 0], 'fro') ** 2 / (N[0] * N[1])
        psnrC1 = 10 * np.log10(maxP**2 / mseC1)
        mseC2 = np.linalg.norm(I[:, :, 1].astype(float) - Xrecovery[:, :, 1], 'fro') ** 2 / (N[0] * N[1])
        psnrC2 = 10 * np.log10(maxP**2 / mseC2)
        mseC3 = np.linalg.norm(I[:, :, 2].astype(float) - Xrecovery[:, :, 2], 'fro') ** 2 / (N[0] * N[1])
        psnrC3 = 10 * np.log10(maxP**2 / mseC3)
        psnrf[iter] = (psnrC1 + psnrC2 + psnrC3)/3

        iter += 1
        print(f"Epoch = {iter}, PSNR = {psnrf[iter-1]:.4f} dB")
        tol = np.linalg.norm((X - last_ten)) / train_norm
        last_ten = X.copy()
        if (tol < epsilon) or (iter >= maxiter):
            ssim_value = ssim(I.astype(float), Xrecovery, data_range=255, channel_axis=2)
            print(f"??PSNR: {psnrf[iter-1]:.4f} dB")
            print(f"??SSIM: {ssim_value:.4f}")
            break
    return Xrecovery, psnrf[:iter], ssim_value

In [ ]:
# 参数设置
seedr = 920
np.random.seed(seedr)

# 图像列表
image_names = ['Airplane', 'House256', 'House512', 'Peppers', 'Tree', 'Sailboat', 'Female']

# 固定参数
missing_rate = 0.80  # 缺失率
lengthscaleU = np.ones(2) * 30
varianceU = np.ones(2)
lengthscaleR = np.ones(2) * 5
varianceR = np.ones(2)
tapering_range = 30
d_MaternU, d_MaternR = 3, 3
R = 10
maxiter, K0 = 100, 70
epsilon = 1e-4

# 参数网格
rho_list = [1, 5, 10, 15, 20]
gamma_list = [0.1, 0.2, 1, 5, 10]

# 创建结果保存目录
os.makedirs(f'results/GLSKF/GLSKF{missing_rate*100}%', exist_ok=True)

# 保存最终结果
results = {}

# 批量处理图像
for img_name in image_names:
    print(f"\n{'='*60}")
    print(f"处理图像: {img_name}")
    print(f"{'='*60}\n")

    try:
        # 读取图像
        I = np.array(Image.open(f'./data/{img_name}.tiff'))
        h, w, c = I.shape

        # 生成缺失掩码
        Omega = np.random.rand(h, w, c) > missing_rate

        print(f"图像尺寸: {I.shape}")
        print(f"缺失率: {missing_rate*100:.1f}%")
        print(f"观测像素: {np.sum(Omega)} / {I.size}\n")

        # 参数搜索
        best_psnr = 0
        best_params = {}
        best_result = None

        print("开始参数网格搜索...")
        print(f"rho 候选: {rho_list}")
        print(f"gamma 候选: {gamma_list}")
        print(f"总共需要测试: {len(rho_list) * len(gamma_list)} 组参数\n")

        param_count = 0
        total_params = len(rho_list) * len(gamma_list)

        for rho in rho_list:
            for gamma in gamma_list:
                param_count += 1
                print(f"\n[{param_count}/{total_params}] 测试参数: rho={rho}, gamma={gamma}")
                print(f"{'-'*50}")

                # 运行算法并记录时间
                start_time = time.perf_counter()
                X, psnr_list, ssim_value = GLSKF(
                    I, Omega, lengthscaleU, lengthscaleR,
                    varianceU, varianceR, tapering_range,
                    d_MaternU, d_MaternR, R, rho, gamma,
                    maxiter, K0, epsilon
                )
                elapsed_time = time.perf_counter() - start_time

                final_psnr = psnr_list[-1]

                print(f"当前结果: PSNR = {final_psnr:.4f} dB, SSIM = {ssim_value:.4f}")
                print(f"运行时间: {elapsed_time:.2f} 秒")

                if final_psnr > best_psnr:
                    best_psnr = final_psnr
                    best_params = {'rho': rho, 'gamma': gamma}
                    best_result = {
                        'recovered': X,
                        'psnr_list': psnr_list,
                        'final_psnr': final_psnr,
                        'ssim': ssim_value,
                        'time': elapsed_time
                    }
                    print(f"*** 发现更好的参数，PSNR 提升至 {best_psnr:.4f} dB ***")

        print(f"\n{'='*60}")
        print(f"最佳参数: rho={best_params['rho']}, gamma={best_params['gamma']}")
        print(f"最佳 PSNR: {best_result['final_psnr']:.4f} dB")
        print(f"最佳 SSIM: {best_result['ssim']:.4f}")
        print(f"最佳时间: {best_result['time']:.2f} 秒")
        print(f"{'='*60}\n")

        results[img_name] = {
            'original': I,
            'observed': I * Omega,
            'recovered': best_result['recovered'],
            'psnr_list': best_result['psnr_list'],
            'final_psnr': best_result['final_psnr'],
            'ssim': best_result['ssim'],
            'time': best_result['time'],
            'best_rho': best_params['rho'],
            'best_gamma': best_params['gamma']
        }

        # 保存图像
        Image.fromarray(np.uint8(best_result['recovered'])).save(
            f'results/GLSKF/GLSKF{missing_rate*100}%/{img_name}_修复后.png'
        )

    except Exception as e:
        print(f"处理 {img_name} 时出错: {str(e)}")
        continue

print(f"\n{'='*60}")
print("所有图像处理完成!")
print(f"{'='*60}")


In [ ]:
# 可视化结果
import pandas as pd
plt.rcParams['font.sans-serif'] = ['SimHei']

for img_name in results.keys():
    data = results[img_name]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(np.uint8(data['original']))
    axes[0].set_title('原始图像', fontsize=14)
    axes[0].axis('off')

    axes[1].imshow(np.uint8(data['observed']))
    axes[1].set_title(f'观测数据 ({missing_rate*100:.0f}% 缺失)', fontsize=14)
    axes[1].axis('off')

    axes[2].imshow(np.uint8(data['recovered']))
    axes[2].set_title(
        f'GLSKF 修复\nPSNR={data["final_psnr"]:.2f}dB, SSIM={data["ssim"]:.4f}\nrho={data["best_rho"]}, gamma={data["best_gamma"]}',
        fontsize=14
    )
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig(f'results/GLSKF/GLSKF{missing_rate*100}%/{img_name}_对比.png', dpi=150, bbox_inches='tight')
    plt.show()

    # PSNR 收敛曲线
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(data['psnr_list']) + 1), data['psnr_list'], 'b-', linewidth=2)
    plt.xlabel('迭代次数', fontsize=12)
    plt.ylabel('PSNR (dB)', fontsize=12)
    plt.title(
        f'{img_name} - GLSKF 收敛曲线\n最佳参数: rho={data["best_rho"]}, gamma={data["best_gamma"]}',
        fontsize=14
    )
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'results/GLSKF/GLSKF{missing_rate*100}%/{img_name}_收敛曲线.png', dpi=150, bbox_inches='tight')
    plt.show()

# 保存结果到 Excel 文件
summary_data = []
for img_name, data in results.items():
    summary_data.append({
        '图像名称': img_name,
        'PSNR (dB)': f"{data['final_psnr']:.4f}",
        'SSIM': f"{data['ssim']:.4f}",
        '运行时间 (s)': f"{data['time']:.2f}",
        '最佳rho': data['best_rho'],
        '最佳gamma': data['best_gamma']
    })

df = pd.DataFrame(summary_data)

# 保存为 Excel
df.to_excel(f'results/GLSKF/GLSKF{missing_rate*100}%/实验总结.xlsx', index=False, engine='openpyxl')

print("\n结果已保存")
print("最佳参数和运行时间已记录在 Excel 文件中")
